# INV-M365-E: Audit Semantics

**Invariants proven:** INV-M365-E-001, INV-M365-E-002, INV-M365-E-003

**Lemmas referenced:** LEM-M365-E-001-01, LEM-M365-E-002-01, LEM-M365-E-003-01

**Phase 1:** docs/math/instruction_contract.md (Sections 4, 5)

## 1. Setup

Deterministic. Success/Failure result. Read-only -> audit=[]; successful Admin mutation -> |audit|=1; failure/rejection -> |audit|=0.

In [ ]:
P = {"Admin", "User"}
A_Admin, A_User = {"admin_mutate", "admin_read"}, {"user_read"}
A_read, A_mut = {"admin_read", "user_read"}, {"admin_mutate"}

def is_valid(instruction):
    if not isinstance(instruction, (tuple, list)) or len(instruction) != 4:
        return False
    p, a = instruction[0], instruction[2]
    return (p == "Admin" and a in A_Admin) or (p == "User" and a in A_User)

def eval_spec(instruction, tenant_state):
    if not is_valid(instruction):
        return None
    p, a = instruction[0], instruction[2]
    if a in A_read:
        return ("Success", tenant_state, [])
    if a in A_mut and p == "Admin":
        return ("Success", tenant_state + 1, [{"event": 1}])
    return None

tau = 0

## 2. Lemma Execution

E-001: Successful Admin mutation -> |audit| = 1.
E-002: Rejected or result failure -> |audit| = 0.
E-003: User or read-only (defined) -> |audit| = 0.

In [ ]:
x_success_mut = ("Admin", "i1", "admin_mutate", {})
x_read = ("User", "i1", "user_read", {})
x_invalid = ("User", "i1", "admin_mutate", {})

out_mut = eval_spec(x_success_mut, tau)
out_read = eval_spec(x_read, tau)
out_inv = eval_spec(x_invalid, tau)

assert out_mut is not None
r, _, audit_mut = out_mut
assert r == "Success" and len(audit_mut) == 1, "E-001 lemma"

assert out_inv is None
assert out_read is not None and len(out_read[2]) == 0, "E-003 lemma"

## 3. Invariant Verification

**INV-M365-E-001:** persona=Admin, action in A_mut, result success -> |audit_events| = 1.

**INV-M365-E-002:** evaluation undefined or result failure -> |audit_events| = 0.

**INV-M365-E-003:** persona=User or action in A_read -> |audit_events| = 0.

In [ ]:
def verify_E001(instruction, tenant_state, eval_fn):
    out = eval_fn(instruction, tenant_state)
    if out is None:
        return
    r, _, audit = out
    if instruction[0] == "Admin" and instruction[2] in A_mut and r == "Success":
        assert len(audit) == 1, "E-001: exactly one audit"

def verify_E002(instruction, tenant_state, eval_fn):
    out = eval_fn(instruction, tenant_state)
    if out is None:
        return
    r, _, audit = out
    if r != "Success":
        assert len(audit) == 0
    if out is None:
        pass

def verify_E003(instruction, tenant_state, eval_fn):
    out = eval_fn(instruction, tenant_state)
    if out is None:
        return
    _, _, audit = out
    if instruction[0] == "User" or instruction[2] in A_read:
        assert len(audit) == 0, "E-003: User/read -> no audit"

verify_E001(x_success_mut, tau, eval_spec)
verify_E002(x_invalid, tau, eval_spec)
verify_E003(x_read, tau, eval_spec)
verify_E003(("Admin", "i1", "admin_read", {}), tau, eval_spec)
print("INV-M365-E-001, E-002, E-003: pass_conditions hold.")

## 4. Failure Demonstration

Rejected instruction produces no output -> no audit. If we had a defined failure result, audit must be [].

In [ ]:
assert eval_spec(x_invalid, tau) is None
assert eval_spec(("Admin", "i1", "user_read", {}), tau) is None
print("Rejected cases produce no audit (undefined -> no triple).")

## 5. Conclusion

INV-M365-E-001, E-002, E-003 hold: one audit per successful Admin mutation; zero for failure/rejection and for User/read-only.

In [ ]:
print("VERIFY: INV-M365-E-001, INV-M365-E-002, INV-M365-E-003 — all pass.")